# Global Terrorism Data Analysis

**Interactive data-analytics project on a synthetic historical incident dataset.**

Every year thousands of terrorism-related incidents occur around the world,
resulting in loss of life, property damage and economic disruption. Analyzing
historical incident records helps researchers, governments and policymakers
identify attack patterns, high-risk regions and long-term trends.

This notebook analyzes a **synthetic incident dataset (2010 - 2024, ~4,000
records)** whose structure mirrors historical terrorism databases such as the
Global Terrorism Database. Country and region geography are real; attacker
organizations are **fictional codenames** (plus a large legitimate *Unclaimed*
share), and all counts are generated by a latent statistical model - the data
is **not** a reproduction of any real database.

- **Data cleaning & preprocessing** - missing values, duplicates, inconsistent
  records, encoded categoricals, normalization, outlier detection.
- **Statistical analysis** - normality tests, t-tests, ANOVA, chi-square,
  point-biserial correlation, correlation heatmap.
- **Exploratory data analysis (EDA)** - bar, line, pie, scatter, histogram,
  box and heatmap charts with **Pandas, NumPy, Matplotlib, Seaborn**.
- **Interactive dashboard** - a self-contained HTML dashboard (Plotly) with a
  geographic map, exported at the end of the notebook.

### Hardware / environment
- Intel Core i5/i7 or AMD Ryzen 5/7, 16 GB RAM, 512 GB SSD.
- Python 3.10+, libraries: Pandas, NumPy, Matplotlib, Seaborn, Plotly.


## Abstract

Terrorist incident data is multidimensional: time, place, method, target and
attacker each vary, and most recorded incidents never get claimed. This project
makes those dimensions visible.

- **Attack frequency** - incidents and casualties per year (2010 - 2024).
- **Geographical distribution** - country-wise and region-wise trends.
- **Attack types** - bombings, armed assaults, assassinations and more, and
  how deadly each method is.
- **Target analysis** - who/what gets targeted most.
- **Casualty statistics** - fatality and injury patterns, rare mass-casualty
  events detected as outliers.
- **Perpetrator patterns** - fictional organization activity vs. unclaimed
  incidents.

The expected outcome is a reproducible analytics system that transforms raw
incident records into meaningful insights supporting academic research, policy
analysis and historical trend evaluation.


## Dataset overview

The generated dataset `terrorism_data.csv` holds one row per incident:

| Column | Meaning |
|---|---|
| IncidentID / Date / Year / Month | Incident identity and time |
| Region / Country / City | Location |
| Lat / Lon | Approximate coordinates (country capital + jitter) |
| AttackType / TargetType / WeaponType | Method, target and weapon category |
| TerroristGroup | Fictional organization or *Unknown / Unclaimed* |
| Fatalities / Injuries | Casualties |
| Suicide (0/1), Success (0/1) | Binary flags |
| PropertyDamage (0/1) | Whether property damage occurred |

The country-weights, year-weights, attack/weapon/target mixes and fatality
model are tuned so the latent structure resembles well-documented trends
(e.g. a 2014 - 2016 global peak, a 2020 dip, concentration in conflict
countries, bombing-heavy casualty profiles).


## Data cleaning & preprocessing

The raw data carries typical production defects:

- Missing City, Lat/Lon, AttackType, TerroristGroup values.
- ~10 duplicated incident records.
- Inconsistent entries: a few negative fatality values.
- Rare mass-casualty events - extreme fatalities detected via the IQR rule.

Cleaning: drop duplicates, impute location from country centroids, fill
categorical misses with modal classes, clip impossible values, and normalize
Fatalities (min-max).


In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_style('whitegrid')

SEED = 42

HAS_SCIPY = True
HAS_PLOTLY = True
try:
    from scipy import stats
except Exception:
    HAS_SCIPY = False
try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    HAS_PLOTLY = False

def show_plotly(fig, h=440, w=None):
    """Render a plotly figure gracefully (needs nbformat for inline display)."""
    if not HAS_PLOTLY:
        print('[Plotly unavailable - skipping figure]')
        return
    layout = dict(height=h)
    if w:
        layout['width'] = w
    fig.update_layout(**layout)
    try:
        fig.show()
    except Exception:
        print('[Inline plotly display needs: pip install nbformat] (run cells in a kernel)')


In [ ]:
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

def pchoice(cat, p):
    if p is not None:
        p = np.asarray(p, dtype=float) / np.sum(p)
    return cat[int(rng.choice(len(cat), p=p))]

# country: (region, relative weight, lat, lon)  - geography is real, weights synthetic
C = {
    'Iraq': ('Middle East & North Africa', 300, 33.3, 44.4),
    'Syria': ('Middle East & North Africa', 170, 34.8, 38.9),
    'Yemen': ('Middle East & North Africa', 90, 15.3, 44.2),
    'Libya': ('Middle East & North Africa', 55, 32.9, 13.2),
    'Egypt': ('Middle East & North Africa', 45, 26.8, 30.8),
    'Turkey': ('Middle East & North Africa', 35, 39.9, 32.9),
    'Lebanon': ('Middle East & North Africa', 20, 33.9, 35.5),
    'Algeria': ('Middle East & North Africa', 18, 36.8, 3.1),
    'Iran': ('Middle East & North Africa', 13, 32.4, 53.7),
    'Israel': ('Middle East & North Africa', 10, 31.5, 34.8),
    'Tunisia': ('Middle East & North Africa', 8, 36.8, 10.2),
    'Jordan': ('Middle East & North Africa', 7, 32.0, 35.9),
    'Saudi Arabia': ('Middle East & North Africa', 7, 24.7, 46.7),
    'Bahrain': ('Middle East & North Africa', 4, 26.1, 50.6),
    'Afghanistan': ('South Asia', 195, 34.5, 69.2),
    'Pakistan': ('South Asia', 150, 30.4, 69.3),
    'India': ('South Asia', 130, 28.6, 77.2),
    'Bangladesh': ('South Asia', 28, 23.7, 90.4),
    'Myanmar': ('South Asia', 22, 21.9, 95.9),
    'Sri Lanka': ('South Asia', 12, 7.3, 80.6),
    'Nepal': ('South Asia', 10, 27.7, 85.3),
    'Philippines': ('Southeast Asia', 90, 14.6, 121.0),
    'Thailand': ('Southeast Asia', 25, 15.9, 101.0),
    'Indonesia': ('Southeast Asia', 18, -6.2, 106.8),
    'Malaysia': ('Southeast Asia', 6, 3.1, 101.7),
    'Nigeria': ('Sub-Saharan Africa', 140, 9.1, 8.7),
    'Somalia': ('Sub-Saharan Africa', 70, 2.1, 45.3),
    'DR Congo': ('Sub-Saharan Africa', 45, -4.3, 15.3),
    'Mali': ('Sub-Saharan Africa', 45, 12.7, -8.0),
    'Cameroon': ('Sub-Saharan Africa', 40, 3.9, 11.5),
    'Burkina Faso': ('Sub-Saharan Africa', 35, 12.4, -1.5),
    'Niger': ('Sub-Saharan Africa', 30, 13.5, 2.1),
    'Sudan': ('Sub-Saharan Africa', 25, 15.6, 32.5),
    'Chad': ('Sub-Saharan Africa', 22, 12.1, 15.0),
    'Kenya': ('Sub-Saharan Africa', 20, -1.3, 36.8),
    'Ethiopia': ('Sub-Saharan Africa', 20, 9.0, 38.7),
    'Mozambique': ('Sub-Saharan Africa', 20, -26.0, 32.6),
    'Central African Republic': ('Sub-Saharan Africa', 18, 4.4, 18.6),
    'South Sudan': ('Sub-Saharan Africa', 15, 4.9, 31.6),
    'Uganda': ('Sub-Saharan Africa', 12, 0.4, 32.6),
    'Cote d Ivoire': ('Sub-Saharan Africa', 7, 5.4, -4.0),
    'Ukraine': ('Europe & Caucasus', 55, 50.5, 30.5),
    'Russia': ('Europe & Caucasus', 25, 55.8, 37.6),
    'France': ('Europe & Caucasus', 15, 48.9, 2.4),
    'United Kingdom': ('Europe & Caucasus', 15, 51.5, -0.1),
    'Germany': ('Europe & Caucasus', 12, 52.5, 13.4),
    'Spain': ('Europe & Caucasus', 8, 40.4, -3.7),
    'Greece': ('Europe & Caucasus', 6, 38.0, 23.7),
    'Italy': ('Europe & Caucasus', 6, 41.9, 12.5),
    'Sweden': ('Europe & Caucasus', 5, 59.3, 18.1),
    'Belgium': ('Europe & Caucasus', 5, 50.9, 4.4),
    'Georgia': ('Europe & Caucasus', 3, 41.7, 44.8),
    'Colombia': ('Latin America & Caribbean', 40, 4.7, -74.1),
    'Mexico': ('Latin America & Caribbean', 30, 23.6, -102.5),
    'Peru': ('Latin America & Caribbean', 8, -12.0, -77.0),
    'Venezuela': ('Latin America & Caribbean', 8, 10.5, -66.9),
    'Ecuador': ('Latin America & Caribbean', 6, -0.2, -78.5),
    'Paraguay': ('Latin America & Caribbean', 5, -23.4, -57.4),
    'Honduras': ('Latin America & Caribbean', 5, 15.2, -86.2),
    'Chile': ('Latin America & Caribbean', 4, -33.4, -70.7),
    'El Salvador': ('Latin America & Caribbean', 4, 13.7, -88.9),
    'Argentina': ('Latin America & Caribbean', 3, -34.6, -58.4),
    'United States': ('North America', 20, 39.8, -98.6),
    'Canada': ('North America', 5, 45.4, -75.7),
    'Kyrgyzstan': ('Central Asia', 3, 42.9, 74.6),
    'Australia': ('Oceania', 2, -25.3, 133.8),
    'New Zealand': ('Oceania', 1, -41.3, 174.8),
}

WY = {2010: 0.80, 2011: 0.88, 2012: 1.00, 2013: 1.30, 2014: 1.55, 2015: 1.65,
      2016: 1.60, 2017: 1.45, 2018: 1.25, 2019: 1.05, 2020: 0.78, 2021: 0.92,
      2022: 1.00, 2023: 1.08, 2024: 1.05}

ATTACKS = ['Bombing / Explosion', 'Armed Assault', 'Assassination',
           'Hostage Taking (Kidnapping)', 'Facility / Infrastructure Attack',
           'Unarmed Assault', 'Hijacking', 'Other']
GLOBAL_AW = [0.34, 0.22, 0.12, 0.10, 0.08, 0.08, 0.02, 0.04]
# regional attack styles (documented tendencies: MENA bomb-heavy, SSA & LatAm
# assault-heavy, SEA kidnapping-heavy, Europe more facility/unarmed, etc.)
ATTACK_W = {
    'Middle East & North Africa': [0.40, 0.20, 0.13, 0.08, 0.08, 0.05, 0.03, 0.03],
    'South Asia': [0.36, 0.20, 0.14, 0.10, 0.07, 0.07, 0.03, 0.03],
    'Southeast Asia': [0.26, 0.20, 0.10, 0.20, 0.08, 0.10, 0.03, 0.03],
    'Sub-Saharan Africa': [0.24, 0.30, 0.10, 0.13, 0.10, 0.07, 0.02, 0.04],
    'Europe & Caucasus': [0.20, 0.22, 0.10, 0.06, 0.12, 0.16, 0.04, 0.10],
    'Latin America & Caribbean': [0.18, 0.30, 0.22, 0.08, 0.08, 0.08, 0.02, 0.04],
    'North America': [0.16, 0.28, 0.08, 0.06, 0.14, 0.18, 0.02, 0.08],
    'Central Asia': [0.30, 0.24, 0.12, 0.10, 0.10, 0.08, 0.02, 0.04],
    'Oceania': [0.20, 0.24, 0.10, 0.08, 0.12, 0.16, 0.02, 0.08],
}

WEAPONS = ['Explosives', 'Firearms', 'Incendiary', 'Melee', 'Vehicle', 'Chemical', 'Other']
W_W = [0.34, 0.44, 0.10, 0.06, 0.03, 0.02, 0.01]

TARGETS = ['Police', 'Government', 'Military', 'Private Citizens & Property',
           'Business', 'Religious Figures / Institutions', 'NGO',
           'Educational Institutions', 'Journalists & Media', 'Tourists', 'Other']
T_W = [0.16, 0.14, 0.15, 0.13, 0.10, 0.08, 0.05, 0.06, 0.04, 0.03, 0.06]

GROUPS = {
    'Middle East & North Africa': ['Crimson Vanguard', 'Iron Crescent',
                                   'Desert Dawn Front', 'Sandstorm Collective',
                                   'Green Dawn Movement', 'Unknown / Unclaimed'],
    'South Asia': ['Highlands Liberation Army', 'Freedom Fields Group',
                   'Red Banner Collective', 'Unknown / Unclaimed'],
    'Southeast Asia': ['Mountain Resistance Group', 'Southern Patriotic Front',
                       'Unknown / Unclaimed'],
    'Sub-Saharan Africa': ['Sahel Resistance Network', 'Horn of Dawn Front',
                           'Coastal Freedom Brigade', 'Baya Liberation Movement',
                           'Unknown / Unclaimed'],
    'Europe & Caucasus': ['Northern Storm Brigade', 'Eastern Dawn Front',
                          'National Vanguard Alliance', 'Unknown / Unclaimed'],
    'Latin America & Caribbean': ['Indigenous Rights Front', 'Blackhawk Brigade',
                                  'National Vanguard Alliance', 'Unknown / Unclaimed'],
    'North America': ['Umbra Collective', 'Crimson Vanguard', 'Unknown / Unclaimed'],
    'Central Asia': ['Umbra Collective', 'Unknown / Unclaimed'],
    'Oceania': ['Umbra Collective', 'National Vanguard Alliance', 'Unknown / Unclaimed'],
}

CAPITALS = {
    'Iraq': 'Baghdad', 'Syria': 'Damascus', 'Yemen': 'Sanaa', 'Libya': 'Tripoli',
    'Egypt': 'Cairo', 'Turkey': 'Ankara', 'Lebanon': 'Beirut', 'Algeria': 'Algiers',
    'Iran': 'Tehran', 'Israel': 'Jerusalem', 'Tunisia': 'Tunis', 'Jordan': 'Amman',
    'Saudi Arabia': 'Riyadh', 'Bahrain': 'Manama', 'Afghanistan': 'Kabul',
    'Pakistan': 'Islamabad', 'India': 'New Delhi', 'Bangladesh': 'Dhaka',
    'Myanmar': 'Naypyidaw', 'Sri Lanka': 'Colombo', 'Nepal': 'Kathmandu',
    'Philippines': 'Manila', 'Thailand': 'Bangkok', 'Indonesia': 'Jakarta',
    'Malaysia': 'Kuala Lumpur', 'Nigeria': 'Abuja', 'Somalia': 'Mogadishu',
    'DR Congo': 'Kinshasa', 'Mali': 'Bamako', 'Cameroon': 'Yaounde',
    'Burkina Faso': 'Ouagadougou', 'Niger': 'Niamey', 'Sudan': 'Khartoum',
    'Chad': "N'Djamena", 'Kenya': 'Nairobi', 'Ethiopia': 'Addis Ababa',
    'Mozambique': 'Maputo', 'Central African Republic': 'Bangui',
    'South Sudan': 'Juba', 'Uganda': 'Kampala', 'Cote d Ivoire': 'Abidjan',
    'Ukraine': 'Kyiv', 'Russia': 'Moscow', 'France': 'Paris',
    'United Kingdom': 'London', 'Germany': 'Berlin', 'Spain': 'Madrid',
    'Greece': 'Athens', 'Italy': 'Rome', 'Sweden': 'Stockholm', 'Belgium': 'Brussels',
    'Georgia': 'Tbilisi', 'Colombia': 'Bogota', 'Mexico': 'Mexico City',
    'Peru': 'Lima', 'Venezuela': 'Caracas', 'Ecuador': 'Quito', 'Paraguay': 'Asuncion',
    'Honduras': 'Tegucigalpa', 'Chile': 'Santiago', 'El Salvador': 'San Salvador',
    'Argentina': 'Buenos Aires', 'United States': 'Washington', 'Canada': 'Ottawa',
    'Kyrgyzstan': 'Bishkek', 'Australia': 'Canberra', 'New Zealand': 'Wellington',
}

N_TXN = 4000
names = list(C)
weights = np.array([C[n][1] for n in names], dtype=float)
weights = weights / weights.sum() * N_TXN
years = list(WY)

rows = []
for i in range(N_TXN):
    country = pchoice(names, weights)
    region, _, clat, clon = C[country]
    year = pchoice(years, [WY[y] for y in years])
    month = int(rng.integers(1, 13))
    day = int(rng.integers(1, 29))
    attack = pchoice(ATTACKS, ATTACK_W.get(region, GLOBAL_AW))
    weapon = pchoice(WEAPONS, W_W)
    target = pchoice(TARGETS, T_W)

    if attack == 'Assassination' and rng.random() < 0.5:
        target = pchoice(['Government', 'Religious Figures / Institutions',
                          'Journalists & Media', 'NGO'], None)
    if attack == 'Hijacking':
        target = 'Other'

    gpool = GROUPS[region]
    gp = [0.5] * len(gpool)
    gp[-1] = 1.0 + 0.7 * rng.random()
    group = pchoice(gpool, gp)

    if rng.random() < 0.012:
        fatal = int(rng.integers(35, 320))
    else:
        fatal = int(round(min(rng.lognormal(0.30, 1.15), 60)))
    if weapon == 'Explosives':
        fatal = int(min(70, round(fatal * 1.35)))

    if weapon == 'Explosives' and attack in ('Bombing / Explosion', 'Vehicle',
                                             'Hijacking', 'Other'):
        suicide = float(rng.random() < 0.18)
    else:
        suicide = float(rng.random() < 0.02)
    if suicide and attack == 'Bombing / Explosion':
        fatal = int(max(fatal, round(rng.lognormal(3.2, 0.9))))

    inj = int(round(min(max(0, fatal * 1.1 * rng.lognormal(0, 0.9) + 1), 900)))
    if fatal >= 150:  # mass-casualty events carry commensurate injuries
        inj = int(round(fatal * rng.uniform(0.7, 1.5)))

    success = float(rng.random() < (0.80 if attack == 'Assassination' else 0.62))
    prop = float(rng.random() < 0.30)

    lat = clat + rng.uniform(-1.6, 1.6)
    lon = clon + rng.uniform(-1.6, 1.6)

    rows.append([i + 1, '%d-%02d-%02d' % (year, month, day), year, month,
                 region, country, CAPITALS[country], round(lat, 3), round(lon, 3),
                 attack, weapon, target, group, fatal, inj, int(suicide),
                 int(success), int(prop)])

df = pd.DataFrame(rows, columns=['IncidentID', 'Date', 'Year', 'Month', 'Region',
                                 'Country', 'City', 'Lat', 'Lon', 'AttackType',
                                 'WeaponType', 'TargetType', 'TerroristGroup',
                                 'Fatalities', 'Injuries', 'Suicide', 'Success',
                                 'PropertyDamage'])

# --- inject dirt --------------------------------------------------------------
for _ in range(30):
    df.loc[rng.integers(0, len(df)), 'City'] = np.nan
for _ in range(14):
    df.loc[rng.integers(0, len(df)), 'Lat'] = np.nan
for _ in range(14):
    df.loc[rng.integers(0, len(df)), 'Lon'] = np.nan
for _ in range(20):
    df.loc[rng.integers(0, len(df)), 'AttackType'] = np.nan
for _ in range(26):
    df.loc[rng.integers(0, len(df)), 'TerroristGroup'] = np.nan
for _ in range(3):
    df.loc[rng.integers(0, len(df)), 'Fatalities'] = -rng.integers(1, 4)
ext_idx = int(rng.integers(0, len(df)))
df.loc[ext_idx, 'Fatalities'] = 980
df.loc[ext_idx, 'Injuries'] = int(980 * rng.uniform(0.7, 1.3))
dup_idx = rng.choice(len(df), size=10, replace=False)
df = pd.concat([df, df.iloc[dup_idx].copy()], ignore_index=True)

df.to_csv('terrorism_data.csv', index=False)
print('Dataset shape:', df.shape)
print(df.head(6).to_string())
print()
print('Rows before cleaning :', len(df))
print('NaNs before cleaning :', int(df.isna().sum().sum()))
print('Duplicates before    :', int(df.duplicated(subset=['IncidentID']).sum()))
print('Regions              :', df['Region'].value_counts().to_dict())
print()
print('Total incidents   :', len(df))
print('Total fatalities  :', int(df['Fatalities'].clip(0, None).sum()))
print('Total injuries    :', int(df['Injuries'].sum()))
print('Peak year         :', int(df.groupby('Year')['IncidentID'].count().idxmax()), '(%d incidents)' %
      df.groupby('Year')['IncidentID'].count().max())


## Descriptive statistics

First pass over the numeric columns - counts, means, spread, missingness, and the class balance of key categorical fields.

In [ ]:
print('Shape:', df.shape, '  |  Columns:', list(df.columns))
print()
print(df[['Fatalities', 'Injuries', 'Year', 'Suicide', 'Success', 'PropertyDamage']]
      .describe().to_string())
print()
print('Missing values per column:')
print(df.isna().sum().to_string())
print()
print('Attack types :')
print(df['AttackType'].value_counts().head(8).to_string())
print()
print('Regions by incidents:')
print(df['Region'].value_counts().head(9).to_string())
print()
print('Top 6 countries:')
print(df['Country'].value_counts().head(6).to_string())


In [ ]:
df = df.drop_duplicates(subset=['IncidentID']).reset_index(drop=True)

# impute Lat/Lon from country centroid
cent = df.groupby('Country')[['Lat', 'Lon']].transform('mean')
df['Lat'] = df['Lat'].fillna(cent['Lat'])
df['Lon'] = df['Lon'].fillna(cent['Lon'])
# impute City from the most common city per country (its stored capital)
cap_city = df.groupby('Country')['City'].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else 'Unknown')
df['City'] = df['City'].fillna(df['Country'].map(cap_city))
df['AttackType'] = df['AttackType'].fillna('Unarmed Assault')
df['TerroristGroup'] = df['TerroristGroup'].fillna('Unknown / Unclaimed')

pre_neg = int((df['Fatalities'] < 0).sum())
df['Fatalities'] = df['Fatalities'].clip(0, None)

q1, q3 = df['Fatalities'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
out_f = int((df['Fatalities'] > upper).sum())
print('Rows after cleaning :', len(df))
print('NaNs after cleaning :', int(df.isna().sum().sum()))
print('Duplicates after    :', int(df.duplicated(subset=['IncidentID']).sum()))
print('Negative fatalities :', pre_neg, 'records clipped to 0')
print('Fatalities IQR upper: %.1f  high outliers: %d' % (upper, out_f))
print('Mass-casualty events:')
print(df.nlargest(6, 'Fatalities')[['Country', 'AttackType', 'Fatalities', 'Injuries']].to_string(index=False))

df['FatalNorm'] = (df['Fatalities'] - df['Fatalities'].min()) / \
                  (df['Fatalities'].max() - df['Fatalities'].min())
df['LogFatal'] = np.log1p(df['Fatalities'])
df['LogInj'] = np.log1p(df['Injuries'])
print()
print('Derived columns       :', ['FatalNorm', 'LogFatal', 'LogInj'])
print('Mean fatalities/attack: %.2f' % df['Fatalities'].mean())
print
print('Median fatalities     :', int(df['Fatalities'].median()))


## Exploratory data analysis (EDA)

Temporal trends, geographic concentration, attack and target mixes, casualties
and perpetrator patterns - with bar, line, pie, scatter, histogram, box and
heatmap charts.


In [ ]:
pals = sns.color_palette('Set2')

# 1. Incidents per year (2010-2024)
yct = df.groupby('Year')['IncidentID'].count()
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(yct.index, yct.values, marker='o', ms=4, color='#2c7fb8', lw=2)
ax.fill_between(yct.index, yct.values, alpha=0.15, color='#2c7fb8')
ax.set_title('Terrorist incidents per year (2010-2024)')
ax.set_xlabel('Year'); ax.set_ylabel('Incidents')
plt.tight_layout(); plt.show()

# 2. Top 15 countries by incidents
t15 = df['Country'].value_counts().head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(t15.index, t15.values, color=pals)
ax.set_title('Top 15 countries by number of incidents')
ax.set_xlabel('Incidents')
for i, v in enumerate(t15.values):
    ax.text(v, i, str(int(v)), va='center', fontsize=8)
plt.tight_layout(); plt.show()

# 3. Attack type distribution
at = df['AttackType'].value_counts()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(at.index, at.values, color=pals)
ax.set_title('Incidents by attack type')
ax.set_ylabel('Incidents')
for i, v in enumerate(at.values):
    ax.text(i, v + 8, str(int(v)), ha='center', fontsize=8)
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()

# 4. Regional share (pie)
reg = df['Region'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(reg.values, labels=reg.index, autopct='%.1f%%', colors=pals,
       startangle=90, textprops={'fontsize': 9})
ax.set_title('Incidents by region')
plt.tight_layout(); plt.show()

# 5. Weapon mix (pie)
fig, ax = plt.subplots(figsize=(7, 5))
wkr = df['WeaponType'].value_counts()
ax.pie(wkr.values, labels=wkr.index, autopct='%.1f%%', colors=pals,
       startangle=90, textprops={'fontsize': 9})
ax.set_title('Incidents by weapon type')
plt.tight_layout(); plt.show()

# 6. Fatalities by year (line)
yf = df.groupby('Year')['Fatalities'].sum()
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(yf.index, yf.values, marker='s', ms=4, color='#a50f15', lw=2)
ax.fill_between(yf.index, yf.values, alpha=0.12, color='#a50f15')
ax.set_title('Total fatalities per year')
ax.set_xlabel('Year'); ax.set_ylabel('Fatalities')
plt.tight_layout(); plt.show()


In [ ]:
pals = sns.color_palette('Set2')

# 7. Top target categories
tg = df['TargetType'].value_counts().head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(tg.index, tg.values, color=pals)
ax.set_title('Top 10 target categories')
ax.set_xlabel('Incidents')
plt.tight_layout(); plt.show()

# 8. Top perpetrator groups (fictional)
gr = df['TerroristGroup'].value_counts().head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(gr.index, gr.values, color=pals)
ax.set_title('Top 10 perpetrator groups (fictional)')
ax.set_xlabel('Incidents')
plt.tight_layout(); plt.show()

# 9. Fatalities distribution (log) + box by weapon
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(df['LogFatal'], bins=25, color='#2c7fb8', edgecolor='white')
axes[0].set_title('Distribution of log1p fatalities')
axes[0].set_xlabel('log1p(fatalities)')
topw = df['WeaponType'].value_counts().head(4).index
axes[1].boxplot([df.loc[df['WeaponType'] == w, 'LogFatal'].values for w in topw])
axes[1].set_xticklabels(topw, rotation=15, ha='right')
axes[1].set_title('Fatalities by weapon (log1p)')
axes[1].set_ylabel('log1p(fatalities)')
plt.tight_layout(); plt.show()

# 10. Suicide vs non-suicide casualties
s1 = df.loc[df['Suicide'] == 1, 'Fatalities'].values
s0 = df.loc[df['Suicide'] == 0, 'Fatalities'].values
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].boxplot([s0, s1], showfliers=False)
axes[0].set_xticklabels(['Non-suicide', 'Suicide'])
axes[0].set_title('Fatalities distribution')
axes[0].set_ylabel('Fatalities')
axes[1].bar(['Non-suicide', 'Suicide'], [s0.mean(), s1.mean()], color=pals)
axes[1].set_title('Mean fatalities per incident')
axes[1].set_ylabel('Mean fatalities')
plt.tight_layout(); plt.show()

# 11. Casualty scatter - fatalities vs injuries
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.scatter(df['Injuries'], df['Fatalities'], s=8, alpha=0.35, color='#2c7fb8')
ax.set_title('Injuries vs fatalities per incident')
ax.set_xlabel('Injuries'); ax.set_ylabel('Fatalities')
plt.tight_layout(); plt.show()

# 12. Region x attack-type heatmap (share of region, %)
ct = pd.crosstab(df['AttackType'], df['Region'])
ctp = (ct / ct.sum(0) * 100)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(ctp, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=.5, ax=ax)
ax.set_title('Attack-type mix by region (%, column-normalized)')
plt.tight_layout(); plt.show()


## Statistical analysis

- **D'Agostino-Pearson test**: distribution of fatalities.
- **Welch t-test**: bombings vs. armed assaults - which is deadlier?
- **ANOVA**: fatalities differ across weapon types.
- **Chi-square**: attack type vs. region (method mixes differ by geography).
- **Point-biserial**: suicide attacks vs. fatalities.
- **Correlation heatmap** of numeric features.


In [ ]:
if HAS_SCIPY:
    f = df['Fatalities'].values
    sh, p = stats.normaltest(f)
    print("1) D'Agostino-Pearson on Fatalities: stat=%.2f  p=%.4g  -> %s" %
          (sh, p, 'NOT normal' if p < 0.05 else 'approx normal'))
    sh2, p2 = stats.normaltest(df['LogFatal'].values)
    print("   D'Agostino-Pearson on LogFatal: stat=%.2f  p=%.4g  -> %s" %
          (sh2, p2, 'NOT normal' if p2 < 0.05 else 'approx normal'))

    bomb = df.loc[df['AttackType'] == 'Bombing / Explosion', 'Fatalities'].values
    assl = df.loc[df['AttackType'] == 'Armed Assault', 'Fatalities'].values
    t, pt = stats.ttest_ind(np.log1p(bomb), np.log1p(assl), equal_var=False)
    print()
    print('2) Welch t-test on log1p fatalities (bombing vs assault): t=%.2f  p=%.4g  -> %s' %
          (t, pt, 'significant difference' if pt < 0.05 else 'no difference'))
    print('   bomb mean=%.2f   assault mean=%.2f  (log scale: %.2f vs %.2f)' %
          (bomb.mean(), assl.mean(), np.log1p(bomb).mean(), np.log1p(assl).mean()))

    wg = [df.loc[df['WeaponType'] == w, 'LogFatal'].values for w in
          df['WeaponType'].value_counts().head(5).index]
    F, pF = stats.f_oneway(*wg)
    print()
    print('3) ANOVA - log-fatalities across weapons: F=%.2f  p=%.4g  -> %s' %
          (F, pF, 'significant weapon effect' if pF < 0.05 else 'no effect'))

    ct = pd.crosstab(df['AttackType'], df['Region'])
    chi, pchi, ddof, _ = stats.chi2_contingency(ct)
    print()
    print('4) Chi-square (attack type x region): chi2=%.2f  dof=%d  p=%.4g  -> %s' %
          (chi, ddof, pchi, 'association' if pchi < 0.05 else 'independent'))

    pb = stats.pointbiserialr(df['Suicide'], df['Fatalities'])
    print()
    print('5) Point-biserial (Suicide vs Fatalities): r=%.3f  p=%.4g' % (pb[0], pb[1]))
    pc = stats.pointbiserialr(df['Success'], df['Fatalities'])
    print('   Point-biserial (Success vs Fatalities): r=%.3f  p=%.4g' % (pc[0], pc[1]))
else:
    print('[SciPy unavailable - statistics skipped]')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[['Fatalities', 'Injuries', 'Suicide', 'Success', 'PropertyDamage',
                'LogFatal', 'LogInj']].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=.5, ax=ax)
ax.set_title('Correlation heatmap')
plt.tight_layout(); plt.show()


## Summary of findings

A short business report of the key terrorism-data numbers produced by the analysis.


In [ ]:
total = len(df)
deaths = int(df['Fatalities'].sum())
inj = int(df['Injuries'].sum())
print('=' * 62)
print('1) Overall picture (2010-2024)')
print('   Incidents .........: %d' % total)
print('   Total fatalities ..: %d' % deaths)
print('   Total injuries ....: %d' % inj)
print('   Avg fatalities ....: %.2f per incident' % (deaths / total))
print('   Success rate ......: %.1f%%' % (100.0 * df['Success'].mean()))
print('   Unclaimed share ...: %.1f%%' %
      (100.0 * (df['TerroristGroup'] == 'Unknown / Unclaimed').mean()))
print()
print('2) Top countries and territories')
for c_, v in df['Country'].value_counts().head(8).items():
    dd = int(df.loc[df['Country'] == c_, 'Fatalities'].sum())
    print('   %-20s : %4d incidents, %4d fatalities' % (c_, v, dd))
print()
print('3) Annual trend')
yct = df.groupby('Year')['IncidentID'].count()
pk = yct.idxmax()
print('   Peak year: %d  (%d incidents)' % (pk, yct[pk]))
print('   Incidents 2015 vs 2020: %d -> %d  (%.0f%% drop)' %
      (yct[2015], yct[2020], 100.0 * (yct[2020] / yct[2015] - 1)))
print('   Incidents 2020 vs 2024: %d -> %d' % (yct[2020], yct[2024]))
print()
print('4) Attack types (deadliest methods)')
atg = df.groupby('AttackType')['Fatalities'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)
for a_, r_ in atg.head(5).iterrows():
    print('   %-28s : avg %.2f  (total %d)' % (a_, r_['mean'], int(r_['sum'])))
print()
print('5) Target mix (top categories)')
for t_, v in df['TargetType'].value_counts().head(6).items():
    print('   %-30s : %4d' % (t_, v))
print()
print('6) Regional profile')
for r_, v in reg.items():
    dd = int(df.loc[df['Region'] == r_, 'Fatalities'].sum())
    print('   %-28s : %4d incidents, %4d fatalities (%.1f%%)' %
          (r_, v, dd, 100.0 * v / total))
print()
print('7) Suicide attacks')
su = df.loc[df['Suicide'] == 1]
print('   %d suicide attacks (+%.1f%% of incidents), avg %.1f fatalities' %
      (len(su), 100.0 * len(su) / total, su['Fatalities'].mean()))


## Interactive dashboard

The final cell exports a fully **self-contained** interactive HTML dashboard (`terrorism_dashboard.html`) built with Plotly - open it in any browser; no server or internet needed. It includes KPIs, trend lines, rankings, a casualty scatter and an offline geographic map.

In [ ]:
if not HAS_PLOTLY:
    raise RuntimeError('Plotly is required to build the dashboard')

yd = df.groupby('Year').agg(Incidents=('IncidentID', 'count'),
                            Fatalities=('Fatalities', 'sum')).reset_index()
fig1 = px.line(yd, x='Year', y='Incidents', title='Incidents per year (2010-2024)',
               markers=True, template='plotly_white')
fig1.update_traces(line_width=2.5, marker_size=5)
fig1.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=35))

c15 = df['Country'].value_counts().head(15).sort_values().reset_index()
c15.columns = ['Country', 'Incidents']
fig2 = px.bar(c15, x='Incidents', y='Country', orientation='h',
              title='Top 15 countries by incidents',
              color='Incidents', color_continuous_scale='YlOrRd',
              template='plotly_white')
fig2.update_layout(height=400, margin=dict(l=10, r=20, t=45, b=10), showlegend=False)

atd = df['AttackType'].value_counts().reset_index()
atd.columns = ['AttackType', 'Count']
fig3 = px.bar(atd, x='AttackType', y='Count', title='Incidents by attack type',
              color='AttackType', color_discrete_sequence=px.colors.qualitative.Set2,
              template='plotly_white')
fig3.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=65), showlegend=False)
fig3.update_xaxes(tickangle=25)

regd = df['Region'].value_counts().reset_index()
regd.columns = ['Region', 'Incidents']
fig4 = px.pie(regd, names='Region', values='Incidents', title='Share of incidents by region',
              color_discrete_sequence=px.colors.qualitative.Set2, template='plotly_white')
fig4.update_layout(height=380, margin=dict(l=15, r=15, t=45, b=15))

grd = df['TerroristGroup'].value_counts().head(12).sort_values().reset_index()
grd.columns = ['Group', 'Incidents']
fig5 = px.bar(grd, x='Incidents', y='Group', orientation='h',
              title='Top 12 perpetrator groups (fictional)',
              color='Incidents', color_continuous_scale='Purples',
              template='plotly_white')
fig5.update_layout(height=400, margin=dict(l=10, r=20, t=45, b=10), showlegend=False)

fig6 = px.line(yd, x='Year', y='Fatalities', title='Total fatalities per year',
               markers=True, template='plotly_white', color_discrete_sequence=['#a50f15'])
fig6.update_traces(line_width=2.5, marker_size=5)
fig6.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=35))

fig7 = px.scatter(df, x='Injuries', y='Fatalities', color='Region',
                  title='Casualties per incident', size='Injuries',
                  hover_name='Country', hover_data=['AttackType'],
                  color_discrete_sequence=px.colors.qualitative.Set2,
                  template='plotly_white')
fig7.update_layout(height=380, margin=dict(l=50, r=20, t=45, b=35), legend_font_size=9)

ca = df.groupby('Country').agg(N=('IncidentID', 'count'), F=('Fatalities', 'sum'),
                               lat=('Lat', 'mean'), lon=('Lon', 'mean')).reset_index()
fig8 = px.scatter_geo(ca, lat='lat', lon='lon', size='N', color='F',
                      hover_name='Country', hover_data={'lat': False, 'lon': False,
                                                        'F': True, 'N': True},
                      title='Geographic distribution of incidents',
                      color_continuous_scale='YlOrRd',
                      projection='natural earth', template='plotly_white')
fig8.update_geos(showland=True, landcolor='#e8efe8', subunitcolor='#bbbbbb',
                 countrycolor='#cccccc')

kpi_html = ('<div class="section"><div class="kpis">'
            '<div class="kpi"><span>Incidents</span><span class="v">' + format(len(df), ",") + '</span></div>'
            '<div class="kpi"><span>Fatalities</span><span class="v">' + format(deaths, ",") + '</span></div>'
            '<div class="kpi"><span>Injured</span><span class="v">' + format(inj, ",") + '</span></div>'
            '<div class="kpi"><span>Success rate</span><span class="v">' + str(round(100.0 * df["Success"].mean(), 1)) + '%</span></div>'
            '<div class="kpi"><span>Peak year</span><span class="v">' + str(int(pk)) + '</span></div>'
            '</div></div>')

pairs = [(fig1, fig2), (fig3, fig4), (fig5, fig6), (fig7, fig8)]
chart_html = ''
first = True
for a, b in pairs:
    if first:
        ha = a.to_html(full_html=False, include_plotlyjs='inline', default_width='100%')
        first = False
    else:
        ha = a.to_html(full_html=False, include_plotlyjs=False, default_width='100%')
    hb = b.to_html(full_html=False, include_plotlyjs=False, default_width='100%')
    chart_html += '<div class="row">' + ha + hb + '</div>'

HEAD = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Terrorism Dashboard</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{font-family:'Segoe UI',Arial,sans-serif;background:#f4f6f9;color:#1d2939;padding:18px}
h1{text-align:center;margin-bottom:4px;color:#0f1b52;font-size:24px}
p.sub{text-align:center;color:#5b6b7f;font-size:13px;margin-bottom:18px}
.note{max-width:1040px;margin:0 auto 14px;background:#fff8e8;border:1px solid #f2d98a;color:#5b4a16;
  font-size:12px;padding:9px 12px;border-radius:8px}
.section{background:#fff;border-radius:12px;box-shadow:0 1px 4px rgba(0,0,0,.08);padding:18px;margin:0 auto 18px;max-width:1180px}
.row{display:flex;flex-wrap:wrap;gap:12px;margin-top:14px}
.row>div{flex:1 1 46%;min-width:330px}
.kpis{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:10px}
.kpi{flex:1 1 150px;background:#eef3fc;border-radius:10px;padding:12px 14px;text-align:center}
.kpi .v{display:block;font-size:20px;font-weight:700;color:#0f1b52;margin-top:3px}
.kpi span{font-size:12px;color:#53606f}
@media(max-width:560px){.row>div{min-width:100%}}
</style>
</head>
<body>
<h1>Global Terrorism Dashboard</h1>
<p class="sub">Synthetic historical incident dataset, 2010-2024 - trends, geography, methods and casualties</p>
<div class="note">Synthetic dataset modeled on historical terrorism-database structure.
Country/region geography is real; attacker organizations are fictional codenames.</div>
"""
TAIL = """
</body>
</html>
"""
with open('terrorism_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(HEAD + kpi_html + chart_html + TAIL)
print('Dashboard written: terrorism_dashboard.html  (%.2f MB)' %
      (os.path.getsize('terrorism_dashboard.html') / 1e6))
print('All figures exported.')
